In [1]:
import json
import glob
import os
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# STEP 1 — MERGE + DEDUPLICATE
# ============================================================

def merge_json_files(folder_path: str) -> list:
    all_posts = []
    files = sorted(glob.glob(f"{folder_path}/*.json"))

    print(f"Found {len(files)} files in {folder_path}")

    for file in files:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  ✅ Loaded {len(data):>4} posts from {os.path.basename(file)}")
        all_posts.extend(data)

    # Deduplicate by URL
    seen = set()
    unique = []
    for post in all_posts:
        url = post.get("url", "")
        if url and url not in seen:
            seen.add(url)
            unique.append(post)

    duplicates_removed = len(all_posts) - len(unique)
    print(f"\n  Total raw posts   : {len(all_posts)}")
    print(f"  Duplicates removed: {duplicates_removed}")
    print(f"  Unique posts      : {len(unique)}")
    return unique

# ============================================================
# STEP 2 — CLEAN EACH POST
# ============================================================

def clean_post(post: dict) -> dict | None:
    text = post.get("text", "").strip()

    # Skip posts with no text at all
    if not text:
        return None

    media = post.get("media", [])

    # Separate images from videos
    image_urls = []
    is_video_post = False

    for item in media:
        typename = item.get("__typename", "")

        if typename == "Photo":
            uri = (
                item.get("image", {}) or {}
            ).get("uri", "")
            if uri:
                image_urls.append(uri)

        elif typename == "Video":
            is_video_post = True

    return {
        "text"          : text,
        "url"           : post.get("url", ""),
        "date"          : post.get("time", ""),
        "likes"         : post.get("likes", 0),
        "shares"        : post.get("shares", 0),
        "is_video_post" : is_video_post,
        "has_images"    : len(image_urls) > 0,
        "image_urls"    : image_urls,
    }

def clean_posts(raw_posts: list) -> tuple[list, dict]:
    cleaned = []
    stats = {
        "total"        : len(raw_posts),
        "no_text"      : 0,
        "video_posts"  : 0,
        "image_posts"  : 0,
        "text_only"    : 0,
    }

    for post in raw_posts:
        result = clean_post(post)
        if result is None:
            stats["no_text"] += 1
            continue
        if result["is_video_post"]:
            stats["video_posts"] += 1
        if result["has_images"]:
            stats["image_posts"] += 1
        else:
            stats["text_only"] += 1
        cleaned.append(result)

    return cleaned, stats

# ============================================================
# STEP 3 — SAVE TO DRIVE
# ============================================================

def save_json(data: list, output_path: str):
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    size_kb = os.path.getsize(output_path) / 1024
    print(f"  💾 Saved {len(data)} posts → {output_path} ({size_kb:.1f} KB)")

# ============================================================
# MAIN — RUN EVERYTHING
# ============================================================

BASE       = "/content/drive/MyDrive/Colab Notebooks/facebook_raw"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/facebook_clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 55)
print("PROCESSING 2025 DATA")
print("=" * 55)
raw_2025            = merge_json_files(f"{BASE}/2025")
clean_2025, stats25 = clean_posts(raw_2025)
save_json(clean_2025, f"{OUTPUT_DIR}/iub_facebook_2025_clean.json")
print(f"\n  📊 2025 Stats: {stats25}")

print("\n" + "=" * 55)
print("PROCESSING 2026 DATA")
print("=" * 55)
raw_2026            = merge_json_files(f"{BASE}/2026")
clean_2026, stats26 = clean_posts(raw_2026)
save_json(clean_2026, f"{OUTPUT_DIR}/iub_facebook_2026_clean.json")
print(f"\n  📊 2026 Stats: {stats26}")

print("\n✅ ALL DONE")
print(f"   2025 clean posts: {len(clean_2025)}")
print(f"   2026 clean posts: {len(clean_2026)}")
print(f"   Grand total     : {len(clean_2025) + len(clean_2026)}")

Mounted at /content/drive
PROCESSING 2025 DATA
Found 6 files in /content/drive/MyDrive/Colab Notebooks/facebook_raw/2025
  ✅ Loaded  600 posts from 09_10-25_07_2025.json
  ✅ Loaded  714 posts from 12_07-24_04_2025.json
  ✅ Loaded  700 posts from 23_04-05_01_2025.json
  ✅ Loaded  157 posts from 23_10-09_10_2025.json
  ✅ Loaded  113 posts from 25_07-12_07_2025.json
  ✅ Loaded  263 posts from 31_12-23_10_2025.json

  Total raw posts   : 2547
  Duplicates removed: 0
  Unique posts      : 2547
  💾 Saved 2386 posts → /content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2025_clean.json (4563.9 KB)

  📊 2025 Stats: {'total': 2547, 'no_text': 161, 'video_posts': 337, 'image_posts': 955, 'text_only': 1431}

PROCESSING 2026 DATA
Found 3 files in /content/drive/MyDrive/Colab Notebooks/facebook_raw/2026
  ✅ Loaded  106 posts from 12_06-14_03_2026.json
  ✅ Loaded  450 posts from 14_03-28_01_2026.json
  ✅ Loaded  450 posts from 28_01-01_01_2026.json

  Total raw posts   : 1006
  Duplica

In [2]:
!sudo apt-get install tesseract-ocr -y
!pip install pytesseract Pillow requests

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [4]:
import json, requests, pytesseract
from PIL import Image
from io import BytesIO

def ocr_image_url(url: str) -> str:
    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return pytesseract.image_to_string(img, lang="eng").strip()
    except Exception as e:
        print(f"OCR failed for {url}: {e}")
        return ""

def enrich_posts_with_ocr(input_path: str, output_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        posts = json.load(f)

    for i, post in enumerate(posts):
        if post.get("has_images") and post.get("image_urls"):
            ocr_texts = [ocr_image_url(url) for url in post["image_urls"]]
            combined_ocr = " ".join(filter(None, ocr_texts))
            post["ocr_text"] = combined_ocr
            post["combined_text"] = (post.get("text", "") + " " + combined_ocr).strip()
        else:
            post["ocr_text"] = ""
            post["combined_text"] = post.get("text", "")

        if i % 100 == 0:
            print(f"Processed {i}/{len(posts)}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(posts, f, ensure_ascii=False, indent=2)
    print(f"Saved enriched file → {output_path}")

# Run for both years
enrich_posts_with_ocr(
    "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2025_clean.json",
    "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2025_enriched.json"
)
enrich_posts_with_ocr(
    "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2026_clean.json",
    "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2026_enriched.json"
)

Processed 0/2386
Processed 100/2386
Processed 200/2386
OCR failed for https://scontent.fcps4-2.fna.fbcdn.net/v/t39.30808-6/541664635_1182103750624567_8715830593941287957_n.jpg?stp=dst-jpg_tt6&cstp=mx1080x1080&ctp=s590x590&_nc_cat=102&ccb=1-7&_nc_sid=127cfc&_nc_ohc=mdnF2gvE3J8Q7kNvwEGhRfP&_nc_oc=AdpiAjnUjuS0Wxa8RYx2_paCgMeP0CidzyYf4LI1H-RBpT0wNnnRAB-RCns1jL3yNP6GcvrU7C2WI0djjnnY-dgQ&_nc_zt=23&_nc_ht=scontent.fcps4-2.fna&_nc_gid=dEI5zsGfnViRN2_k6asqPg&_nc_ss=72289&oh=00_Af-jfP9y8mkHBMmpzowWnfG9Rg5auNGBsXGMvt30JTWE7w&oe=6A31646D: HTTPSConnectionPool(host='scontent.fcps4-2.fna.fbcdn.net', port=443): Max retries exceeded with url: /v/t39.30808-6/541664635_1182103750624567_8715830593941287957_n.jpg?stp=dst-jpg_tt6&cstp=mx1080x1080&ctp=s590x590&_nc_cat=102&ccb=1-7&_nc_sid=127cfc&_nc_ohc=mdnF2gvE3J8Q7kNvwEGhRfP&_nc_oc=AdpiAjnUjuS0Wxa8RYx2_paCgMeP0CidzyYf4LI1H-RBpT0wNnnRAB-RCns1jL3yNP6GcvrU7C2WI0djjnnY-dgQ&_nc_zt=23&_nc_ht=scontent.fcps4-2.fna&_nc_gid=dEI5zsGfnViRN2_k6asqPg&_nc_ss=72289&oh=00_A

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
def merge_enriched_files(paths: list[str], output_path: str):
    all_posts = []
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            all_posts.extend(data)

    # Deduplicate by URL just in case
    seen = set()
    unique_posts = []
    for post in all_posts:
        url = post.get("url", "")
        if url not in seen:
            seen.add(url)
            unique_posts.append(post)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(unique_posts, f, ensure_ascii=False, indent=2)
    print(f"Merged total: {len(unique_posts)} posts → {output_path}")

merge_enriched_files(
    [
        "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2025_enriched.json",
        "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_2026_enriched.json",
    ],
    "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_all_enriched.json"
)

Merged total: 3308 posts → /content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_all_enriched.json


In [8]:
def chunk_text(text: str, max_chars: int = 1500) -> list[str]:
    """Simple sentence-aware chunker."""
    sentences = text.replace("\n", " ").split(". ")
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) < max_chars:
            current += sentence + ". "
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sentence + ". "
    if current.strip():
        chunks.append(current.strip())
    return chunks if chunks else [text]

def prepare_chunks(merged_path: str) -> list[dict]:
    with open(merged_path, "r", encoding="utf-8") as f:
        posts = json.load(f)

    all_chunks = []
    for post in posts:
        text = post.get("combined_text", "").strip()
        if not text:
            continue
        year = "2026" if "2026" in post.get("url", "") else "2025"
        for idx, chunk in enumerate(chunk_text(text)):
            all_chunks.append({
                "text": chunk,
                "metadata": {
                    "source": "facebook",
                    "year": year,
                    "url": post.get("url", ""),
                    "likes": post.get("likes", 0),
                    "chunk_index": idx,
                }
            })

    print(f"Total chunks ready for embedding: {len(all_chunks)}")
    return all_chunks

chunks = prepare_chunks(
    "/content/drive/MyDrive/Colab Notebooks/facebook_clean/iub_facebook_all_enriched.json"
)

Total chunks ready for embedding: 3921


In [ ]:
!pip install chromadb google-generativeai

import chromadb
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

# Initialize persistent ChromaDB
chroma_client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/IUB_Chatbot/chromadb"
)
collection = chroma_client.get_or_create_collection(
    name="iub_facebook_posts",
    metadata={"hnsw:space": "cosine"}
)

def embed_batch(texts: list[str]) -> list[list[float]]:
    result = genai.embed_content(
        model="models/text-embedding-004",
        content=texts,
        task_type="retrieval_document"
    )
    return result["embedding"]

# Embed in batches of 100 (API limit)
BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    texts   = [c["text"] for c in batch]
    ids     = [f"chunk_{i + j}" for j in range(len(batch))]
    metas   = [c["metadata"] for c in batch]

    embeddings = embed_batch(texts)

    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metas,
        ids=ids
    )
    print(f"Embedded {i + len(batch)}/{len(chunks)} chunks")

print("✅ All chunks embedded into ChromaDB!")